# Single-mutation effect sizes and significance

Background-averaged effect of each of the ten substitutions in every library, computed as the mean paired difference in affinity between genotypes differing only at that site. Significance comes from a Wilcoxon signed-rank test on those paired differences, corrected once across all 180 tests (9 antibodies, 2 antigens, 10 sites) by Benjamini-Hochberg rather than per library.

Also reports, per site, the fraction of pairs touching a floor-censored genotype. Those deltas are biased toward zero, so the column flags where an effect size is a lower bound.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

data_dir = "../data"
table_dir = "../tables"
os.makedirs(table_dir, exist_ok=True)

numbering       = "kabat"  # "kabat" or "imgt"
detection_floor = 6.0      # left-censoring floor in -log10(Kd); used to flag floor-biased deltas

# display name to file stem; the hyphens and the dot do not appear in any filename
name_to_stem = {'1NC9':'1NC9','CH235.12':'CH235_12','N49P7':'N49P7','NIH45-46':'NIH4546',
                'VRC-PG20':'VRCPG20','VRC01':'VRC01','VRC07':'VRC07','VRC13':'VRC13',
                'VRC16':'VRC16'}

# Helpers

In [ ]:
# G -> '0' (germline / ancestral), M -> '1' (mature / derived)
def genotype_to_binary(row, site_cols):
    return "".join("1" if row[c] == "M" else "0" for c in site_cols)

# Mean -log10(Kd) per (genotype, antigen) from raw measurements. Data is
# left-censored at detection_floor (6.0); floored values are kept as measured,
# not imputed, so deltas involving a floored genotype are lower bounds.
def process_raw_data(df_raw):
    site_cols = [c for c in df_raw.columns if c.startswith("site_")]
    df_raw["geno"] = df_raw.apply(lambda row: genotype_to_binary(row, site_cols), axis=1)
    df_raw["variant_int"] = df_raw["geno"].apply(lambda x: int(x, 2))
    mean_kd = (df_raw.groupby(["variant_int", "geno", "antigen"])["nlog10_Kd"]
               .mean().reset_index())
    return mean_kd

# Site labels ordered by site index, e.g. S56R (germline-position-mature)
def load_site_labels(antibody, scheme=numbering):
    meta = pd.read_csv(f"{data_dir}/{name_to_stem[antibody]}_meta.csv")
    order = meta["site"].str.replace("site_", "", regex=False).astype(int)
    meta = meta.assign(_i=order).sort_values("_i")
    pos_col = "kabat_position" if scheme == "kabat" else "imgt_position"
    return [f"{r.germline_aa}{getattr(r, pos_col)}{r.mature_aa}" for r in meta.itertuples()]

# Delta (mut - bg) across all backgrounds per site, plus the fraction of finite
# pairs that touch a floored genotype (mean at detection_floor). Those pairs are
# the deltas biased toward zero by left-censoring.
def mutation_effect_distributions(mean_kd, antigen, n_sites=10):
    sub = mean_kd[mean_kd.antigen == antigen]
    aff = (sub.set_index("variant_int")["nlog10_Kd"]
              .reindex(range(2 ** n_sites))
              .to_numpy())                       # NaN where a genotype was not measured
    floored = np.isclose(aff, detection_floor)   # NaN compares False
    idx = np.arange(2 ** n_sites)
    deltas, cens_frac = {}, {}
    for i in range(n_sites):
        mask = 1 << (n_sites - 1 - i)
        bg   = idx[(idx & mask) == 0]            # backgrounds with site_i germline
        mut  = bg | mask                         # same background, site_i matured
        d    = aff[mut] - aff[bg]
        keep = np.isfinite(d)
        deltas[i] = d[keep]
        touched = (floored[bg] | floored[mut])[keep]
        cens_frac[i] = float(touched.mean()) if keep.any() else np.nan
    return deltas, cens_frac

# Wilcoxon signed-rank p and Cohen's d per site. No correction here:
# BH is applied once globally across all complexes (see the build cell).
def cohens_d(arr):
    return arr.mean() / arr.std()

def site_raw_stats(deltas, n_sites=10):
    raw, effect_d = [], []
    for i in range(n_sites):
        d = deltas[i]
        _, p = wilcoxon(d, alternative="two-sided")
        raw.append(p)
        effect_d.append(cohens_d(d))
    return raw, effect_d

# Compute per-site d and p (global BH)

In [ ]:
antibodies = ["1NC9", "CH235.12", "N49P7", "NIH45-46", "VRC-PG20", "VRC01", "VRC07", "VRC13", "VRC16"]

# Stage 1: per-site deltas and censored fraction for every complex
all_results = {}
labels_by_ab = {}
for ab in antibodies:
    df_raw = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_kds.csv")
    labels_by_ab[ab] = load_site_labels(ab)
    res = {}
    for ag in sorted(df_raw.antigen.unique()):
        mean_kd = process_raw_data(df_raw[df_raw.antigen == ag].copy())
        res[ag] = mutation_effect_distributions(mean_kd, ag)
    all_results[ab] = res

# Stage 2: one global Benjamini-Hochberg family across every (antibody, antigen, site)
recs = []
for ab, res_by_ag in all_results.items():
    labels = labels_by_ab[ab]
    for ag, (deltas, cens) in res_by_ag.items():
        raw, effect_d = site_raw_stats(deltas, n_sites=len(deltas))
        for i in range(len(deltas)):
            recs.append({"antibody": ab, "antigen": ag, "site_index": i,
                         "site_label": labels[i], "n": deltas[i].size,
                         "d": effect_d[i], "censored_frac": cens[i], "p_raw": raw[i]})

stats_all = pd.DataFrame(recs)
mask = stats_all["p_raw"].notna()
stats_all["p"] = np.nan
_, corrected, _, _ = multipletests(stats_all.loc[mask, "p_raw"].values,
                                   alpha=0.05, method="fdr_bh")
stats_all.loc[mask, "p"] = corrected
stats_all = stats_all[["antibody", "antigen", "site_index", "site_label",
                       "n", "d", "censored_frac", "p_raw", "p"]]

stats_all.to_csv(f"{table_dir}/site_statistics.csv", index=False)
print(f"global BH family: {int(mask.sum())} tests across "
      f"{stats_all.groupby(['antibody','antigen']).ngroups} complexes")
print(stats_all.head())